# Netz Temporal Stroke Router (ROUTER-002)

**Policy: HARD LANDING.** Real human strokes only. No synthetic generators.

**Data sources (auto where legally possible; turn Settings -> Internet ON):**
- NUMBER / MATH: official Google **MathWriting** is downloaded automatically (human-written train/valid/test only).
- OBJECT: real doodles from Google **Quick, Draw!** are streamed automatically (CC BY 4.0). Disable with `NETZ_QUICKDRAW=0`.
- TEXT: **IAM-OnDB is licence-gated and cannot be auto-downloaded.** One-time setup: register (free) at fki.tic.heia-fr.ch, upload `lineStrokes-all.tar.gz` + `ascii-all.tar.gz` as a *private* Kaggle dataset, then attach it or set `NETZ_IAM_KAGGLE=yourname/slug`. The notebook stops with exact instructions if TEXT/MATH are missing (`NETZ_ALLOW_PARTIAL=1` overrides).

### What changed vs. v2
- **Shortcut protection:** the router now sees **geometry only** (`x, y, dx, dy, stroke_start, stroke_end`; the timing columns are masked inside the model). Previously `dt`/`speed` let it learn *which dataset* a sample came from (IAM vs MathWriting sampling rates) instead of *what* was written.
- **Random stroke-crop augmentation** (prefix windows = "in-progress" ink, plus windows in the middle) so TEXT and MATH look alike in size and stroke count, and so the router works on partial input - which is what the app will actually feed it.
- **Honest classes:** classes without real data (MIXED / OBJECT / UNKNOWN) are masked out of the exported model instead of silently outputting garbage. Provide them via `router_data.jsonl`, or opt in to `NETZ_COMPOSE_MIXED=1` (MIXED built by placing a real text crop next to a real math crop - **off by default** because it is composed data).
- **Cleaner labels:** MathWriting samples that are purely alphabetic (look like TEXT) are dropped; shuffled, capped, balanced via class-weighted loss.
- **Leak-safe splits** (IAM by form id, MathWriting by official folder or file hash), val **macro-F1** model selection, per-class precision/recall/F1 + confusion matrix, evaluation on full **and** cropped ink.
- **Shortcut detector:** a tiny classifier using only trivial stats (stroke count, aspect ratio) is trained; if it scores nearly as well as the network, you get a warning.
- ONNX takes the same canonical 8D features as the other two models and is verified against PyTorch.

### How to run on Kaggle
1. **File -> Import Notebook** -> `train_router.ipynb`
2. **+ Add Input**: `iam-ondb` (TEXT) **and** `mathwriting` (NUMBER/MATH), or your own `router_data.jsonl`.
3. Accelerator: GPU T4/P100 (CPU is fine too). **Run All**.
4. Download `netz_router.onnx`, `netz_router_meta.json`, `netz_router_feature_test.json` into `Netz/public/models/`.

`router_data.jsonl` format: one JSON per line: `{"strokes": [[{"x":..,"y":..,"t":..}, ...], ...], "label": "TEXT|NUMBER|MATH|MIXED|OBJECT|UNKNOWN"}`
Env overrides: `NETZ_IAM_DIR`, `NETZ_MATH_DIR`, `NETZ_EPOCHS`, `NETZ_PER_CLASS`, `NETZ_COMPOSE_MIXED`.

In [ ]:
# 1. Environment & configuration
import os, sys, re, math, json, time, random, hashlib, copy, subprocess
import xml.etree.ElementTree as ET
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def _pip(pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
try:
    import onnx, onnxruntime as ort
except ImportError:
    _pip(['onnx', 'onnxruntime'])
    import onnx, onnxruntime as ort

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ROUTER_CLASSES = ['TEXT', 'NUMBER', 'MATH', 'MIXED', 'OBJECT', 'UNKNOWN']
CLASS_TO_IDX = {c: i for i, c in enumerate(ROUTER_CLASSES)}
CFG = dict(
    max_len=256,
    epochs=int(os.environ.get('NETZ_EPOCHS', 40)),
    batch_size=64, lr=2e-3, patience=8,
    per_class=int(os.environ.get('NETZ_PER_CLASS', 4000)),
    max_math_scan=60000,
    compose_mixed=os.environ.get('NETZ_COMPOSE_MIXED', '0') == '1',
    quickdraw=os.environ.get('NETZ_QUICKDRAW', '1') == '1',        # real OBJECT strokes from Google Quick, Draw! (CC BY 4.0)
    num_workers=int(os.environ.get('NETZ_WORKERS', 2)),
    scale_mode='max',
)
print(f'Router environment: {device} | PyTorch {torch.__version__}')
print('Config:', CFG)

# ======================================================================
# Netz canonical stroke features  --  IDENTICAL in all three notebooks.
# Mirror this exactly in the web app (JS) and verify with the exported
# *_feature_test.json file.
#
# Raw ink array: float32 (N, 4) = [x, y, t, stroke_id]
# Feature columns (8): 0 x | 1 y | 2 dx | 3 dy | 4 dt | 5 stroke_end | 6 stroke_start | 7 speed
# ======================================================================
FEATURE_DIM = 8
FEATURE_SPEC = ['x', 'y', 'dx', 'dy', 'dt', 'stroke_end', 'stroke_start', 'speed']


def strokes_to_array(strokes):
    """strokes: list of strokes; a point is dict(x,y[,t]) or a sequence (x, y[, t])."""
    rows = []
    for sid, stroke in enumerate(strokes):
        for pt in stroke:
            if isinstance(pt, dict):
                x, y, t = pt['x'], pt['y'], pt.get('t', None)
            else:
                x, y = pt[0], pt[1]
                t = pt[2] if len(pt) >= 3 else None
            rows.append((float(x), float(y), np.nan if t is None else float(t), float(sid)))
    arr = np.asarray(rows, dtype=np.float64).reshape(-1, 4)
    if len(arr) and np.isfinite(arr[:, 2]).any():
        arr[:, 2] -= np.nanmin(arr[:, 2])          # relative time keeps float32 precise
    return arr.astype(np.float32)


def _to_seconds(t):
    """Auto-detect ms vs s from the median sampling gap; fall back to 10 ms/sample if no timing."""
    t = np.asarray(t, dtype=np.float64)
    n = len(t)
    fallback = np.arange(n, dtype=np.float64) * 0.01
    if n < 2 or not np.isfinite(t).all():
        return fallback
    d = np.diff(t)
    pos = d[d > 0]
    if len(pos) == 0:
        return fallback
    return t / 1000.0 if np.median(pos) > 1.0 else t


def extract_features(arr, max_len, scale_mode='max'):
    """Return float32 (n<=max_len, 8). No padding here (batching pads and tracks real lengths).
    scale_mode: 'max'    -> divide by max(width, height)      (math, router)
                'height' -> divide by max(height, 0.03*width) (text lines)"""
    arr = np.asarray(arr)
    n = len(arr)
    if n == 0:
        return np.zeros((1, FEATURE_DIM), dtype=np.float32)
    t_sec = _to_seconds(arr[:, 2])
    if n > max_len:                                   # deterministic, JS-reproducible subsampling
        idx = np.floor(np.arange(max_len) * (n - 1) / (max_len - 1) + 0.5).astype(np.int64)
        arr, t_sec = arr[idx], t_sec[idx]
        n = max_len
    x = arr[:, 0].astype(np.float64)
    y = arr[:, 1].astype(np.float64)
    sid = arr[:, 3]
    x0, y0 = x.min(), y.min()
    w, h = x.max() - x0, y.max() - y0
    scale = max(h, 0.03 * w, 1e-6) if scale_mode == 'height' else max(w, h, 1e-6)
    xn, yn = (x - x0) / scale, (y - y0) / scale
    dx = np.diff(xn, prepend=xn[0])
    dy = np.diff(yn, prepend=yn[0])
    dt = np.diff(t_sec, prepend=t_sec[0])
    dt = np.where(np.arange(n) == 0, 0.0, np.maximum(dt, 1e-3))
    speed = np.sqrt(dx * dx + dy * dy) / np.maximum(dt, 1e-3)
    start = np.ones(n)
    start[1:] = (sid[1:] != sid[:-1])
    end = np.ones(n)
    end[:-1] = (sid[:-1] != sid[1:])
    feats = np.stack([xn, yn, dx, dy, np.minimum(dt * 10.0, 1.0), end, start, np.minimum(speed / 5.0, 1.0)], axis=1)
    return feats.astype(np.float32)


# ---------------- augmentation on raw ink (train only) ----------------
def thin_points(arr, rng, keep_min=0.7):
    """Randomly drop points (simulates other sampling rates); stroke endpoints are always kept."""
    n = len(arr)
    if n < 8:
        return arr
    sid = arr[:, 3]
    keep = rng.random(n) < rng.uniform(keep_min, 1.0)
    keep[0] = keep[-1] = True
    keep[1:] |= (sid[1:] != sid[:-1])
    keep[:-1] |= (sid[:-1] != sid[1:])
    return arr[keep]


def augment_array(arr, rng, thin=True):
    a = np.array(arr, dtype=np.float64)
    if len(a) == 0:
        return arr
    if thin:
        a = thin_points(a, rng)
    x, y = a[:, 0], a[:, 1]
    cx, cy = x.mean(), y.mean()
    ang = rng.uniform(-0.08, 0.08)
    sx, sy = rng.uniform(0.85, 1.15), rng.uniform(0.85, 1.15)
    shear = rng.uniform(-0.25, 0.25)
    xr, yr = x - cx, y - cy
    x2 = xr * np.cos(ang) - yr * np.sin(ang)
    y2 = xr * np.sin(ang) + yr * np.cos(ang)
    x2 = x2 + shear * y2
    a[:, 0] = x2 * sx + cx
    a[:, 1] = y2 * sy + cy
    span = max(np.ptp(a[:, 0]), np.ptp(a[:, 1]), 1e-6)
    a[:, :2] += rng.normal(0.0, 0.002 * span, size=(len(a), 2))
    return a.astype(np.float32)


# ---------------- shared metrics ----------------
def edit_distance(a, b):
    """Levenshtein distance between two sequences (strings or lists)."""
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


# ---------------- parity test vector for the JS port ----------------
TEST_STROKES = [
    # case 1: tiny, ms timestamps, pen-up gap between the two strokes
    [[[0, 0, 0], [10, 5, 10], [20, 8, 20], [30, 20, 30]],
     [[5, 40, 100], [15, 42, 110], [25, 50, 120]]],
    # case 2: 3000 points -> exercises the subsampling path
    [[[i, round(100 * math.sin(i / 50), 3), i * 10] for i in range(1500)],
     [[i, round(100 * math.cos(i / 50), 3) + 300, 20000 + i * 10] for i in range(1500)]],
]


def write_feature_test(path, max_len, scale_mode):
    cases = []
    for strokes in TEST_STROKES:
        f = extract_features(strokes_to_array(strokes), max_len, scale_mode)
        cases.append({'strokes': strokes, 'features': np.round(f, 6).tolist()})
    with open(path, 'w') as fh:
        json.dump({'feature_spec': FEATURE_SPEC, 'max_len': max_len, 'scale_mode': scale_mode,
                   'tolerance': 1e-4, 'cases': cases}, fh)
    print(f'Wrote {path}  (feed these strokes through the JS extractor; max abs diff must be < 1e-4)')

# ---------------- dataset discovery helpers (archives + diagnostics) ----------------
import tarfile, zipfile
from collections import Counter

def list_inputs(max_dirs=40):
    """Print what is actually attached under /kaggle/input so a failed search is easy to debug."""
    base = '/kaggle/input'
    if not os.path.isdir(base):
        print('No /kaggle/input here (not running on Kaggle?). Set NETZ_IAM_DIR / NETZ_MATH_DIR to your data folders.')
        return
    print('Attached inputs:', sorted(os.listdir(base)) or 'NONE - click "+ Add Input" in the right sidebar')
    shown = 0
    for dp, dn, fn in os.walk(base):
        if dp[len(base):].count(os.sep) > 4:
            dn[:] = []
            continue
        exts = Counter(os.path.splitext(f)[1].lower() or '(none)' for f in fn)
        print(f'  {dp}   subdirs={len(dn)} files={len(fn)} {dict(exts.most_common(4))}')
        shown += 1
        if shown >= max_dirs:
            print('  ...'); break

def extract_archives(src_roots, dest=None):
    """Extract only the ink/transcript files from .zip/.tar/.tar.gz/.tgz found in src_roots (skips 'synthetic')."""
    wanted = ('.inkml', '.xml', '.txt', '.jsonl', '.mlf')
    if dest is None:
        dest = '/kaggle/working/netz_extracted' if os.path.isdir('/kaggle/working') else os.path.expanduser('~/.cache/netz/extracted')
    out = []
    for root in src_roots:
        for dp, _, fn in os.walk(root):
            for f in fn:
                low = f.lower()
                if not low.endswith(('.tar.gz', '.tgz', '.tar', '.zip')):
                    continue
                target = os.path.join(dest, re.sub(r'\W+', '_', f))
                full = os.path.join(dp, f)
                if not os.path.isdir(target):
                    os.makedirs(target, exist_ok=True)
                    print(f'Extracting {full} -> {target} (ink/transcript files only)')
                    try:
                        if low.endswith('.zip'):
                            with zipfile.ZipFile(full) as zf:
                                names = [n for n in zf.namelist() if n.lower().endswith(wanted) and 'synthetic' not in n.lower()]
                                zf.extractall(target, members=names)
                        else:
                            with tarfile.open(full) as tf:
                                members = [m for m in tf if m.isfile() and m.name.lower().endswith(wanted) and 'synthetic' not in m.name.lower()]
                                try:
                                    tf.extractall(target, members=members, filter='data')
                                except TypeError:
                                    tf.extractall(target, members=members)
                    except Exception as e:
                        print('  could not extract:', e)
                out.append(target)
    return out


MATHWRITING_URL = os.environ.get('NETZ_MW_URL', 'https://storage.googleapis.com/mathwriting_data/mathwriting-2024.tgz')   # official, 2.9 GB

def download_mathwriting(dest='/kaggle/working/mathwriting_real'):
    """Official Google MathWriting (needs Internet: ON). Streams the .tgz and keeps ONLY human-written train/valid/test InkML
    (synthetic and symbols are skipped, which also saves ~GBs of disk). Returns the folder or None."""
    import urllib.request
    if not os.path.isdir('/kaggle/working'):
        dest = os.path.expanduser('~/.cache/netz/mathwriting_real')
    if os.path.isdir(dest) and any(f.endswith('.inkml') for _, _, fs in os.walk(dest) for f in fs):
        return dest
    os.makedirs(dest, exist_ok=True)
    tgz = os.path.join(dest, 'mathwriting.tgz')
    try:
        print(f'Downloading MathWriting from {MATHWRITING_URL} (2.9 GB, takes a few minutes) ...')
        urllib.request.urlretrieve(MATHWRITING_URL, tgz, _progress_hook())
        n = 0
        with tarfile.open(tgz, 'r|gz') as tf:                       # streaming: no seeking, low memory
            for m in tf:
                parts = [p.lower() for p in m.name.split('/')]
                if m.isfile() and m.name.lower().endswith('.inkml') and any(p in ('train', 'valid', 'test') for p in parts[:-1]) \
                        and not any(p in ('synthetic', 'symbols') for p in parts):
                    tf.extract(m, dest, **({'filter': 'data'} if hasattr(tarfile, 'data_filter') else {}))
                    n += 1
        os.remove(tgz)
        print(f'Extracted {n} human-written InkML files to {dest}')
        return dest if n else None
    except Exception as e:
        print(f'MathWriting download failed: {e}\n  -> turn Internet ON in Settings, or attach MathWriting as an input.')
        return None


def _progress_hook():
    state = {'next': 0}
    def hook(blocks, bsize, total):
        if total > 0:
            pct = min(100, int(blocks * bsize * 100 / total))
            if pct >= state['next']:
                print(f'   download {pct}%', flush=True)
                state['next'] = pct + 10
    return hook

IAM_KAGGLE_HANDLE = os.environ.get('NETZ_IAM_KAGGLE', '')     # e.g. 'yourname/iam-ondb' = YOUR private upload of IAM-OnDB
IAM_URLS = [u for u in os.environ.get('NETZ_IAM_URLS', '').split(',') if u.strip()]   # direct links to lineStrokes-all.tar.gz, ascii-all.tar.gz, ...

def download_iam(dest='/kaggle/working/iam_downloaded'):
    """IAM-OnDB is licence-gated (free registration at fki.tic.heia-fr.ch), so there is NO public auto-download.
    This fetches it from a place YOU control: a private Kaggle dataset (NETZ_IAM_KAGGLE) or direct URLs (NETZ_IAM_URLS)."""
    import urllib.request
    if not os.path.isdir('/kaggle/working'):
        dest = os.path.expanduser('~/.cache/netz/iam_downloaded')
    os.makedirs(dest, exist_ok=True)
    got = []
    if IAM_KAGGLE_HANDLE:
        try:
            import kagglehub
            print(f'Downloading your IAM-OnDB upload from Kaggle: {IAM_KAGGLE_HANDLE}')
            got.append(kagglehub.dataset_download(IAM_KAGGLE_HANDLE))
        except Exception as e:
            print(f'  kagglehub download failed: {e}')
    for url in IAM_URLS:
        try:
            fn = os.path.join(dest, os.path.basename(url.split('?')[0]) or 'iam_file')
            print(f'Downloading {url}')
            urllib.request.urlretrieve(url.strip(), fn, _progress_hook())
            got.append(dest)
        except Exception as e:
            print(f'  download failed: {e}')
    return sorted(set(got))

IAM_HELP = (
    'IAM-OnDB cannot be auto-downloaded (licence: free registration required).\n'
    '  ONE-TIME SETUP, then it is automatic forever:\n'
    '   1. (The old unibe.ch address is retired.) Register (free) at https://fki.tic.heia-fr.ch/databases/iam-on-line-handwriting-database and download the On-Line DB files:\n'
    '      lineStrokes-all.tar.gz, ascii-all.tar.gz and the 4 split files trainset.txt, testset_v.txt, testset_t.txt, testset_f.txt\n'
    '   2. Kaggle -> Datasets -> New Dataset -> upload those files -> keep it PRIVATE.\n'
    '   3. In this notebook either click "+ Add Input" and attach it, OR set  NETZ_IAM_KAGGLE=yourname/your-dataset-slug  (auto-download).')

In [ ]:
# 2. Helpers: splits, IAM + InkML readers
def hash_split(key):
    v = int(hashlib.md5(key.encode()).hexdigest(), 16) % 100
    return 'train' if v < 80 else ('val' if v < 90 else 'test')

LINE_XML_RE = re.compile(r'^([a-z]\d{2}-\d{3}[a-z]?)-(\d{2})\.xml$', re.I)
EXCLUDE_DIRS = ('synthetic', 'symbols', 'symbol')
NUM_RE = re.compile(r'[-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?')

def path_split_math(path):
    parts = [p.lower() for p in re.split(r'[\\/]', path)]
    if any(p in EXCLUDE_DIRS for p in parts):
        return 'EXCLUDED'
    for p in parts[:-1]:
        if p == 'train': return 'train'
        if p in ('valid', 'val', 'validation'): return 'val'
        if p == 'test': return 'test'
    return None

def parse_iam_xml(path):
    root = ET.parse(path).getroot()
    strokes = []
    for st in root.iter('Stroke'):
        pts = [(float(p.get('x')), float(p.get('y')), float(p.get('time', 'nan'))) for p in st.iter('Point')]
        if pts: strokes.append(pts)
    return strokes

def parse_inkml(path):
    root = ET.parse(path).getroot()
    strokes, norm, raw, bad = [], None, None, False
    for el in root.iter():
        tag = el.tag.split('}')[-1]
        if tag == 'trace':
            text = el.text or ''
            if "'" in text or '"' in text:
                bad = True; continue
            pts = []
            for seg in text.split(','):
                nums = NUM_RE.findall(seg)
                if len(nums) >= 3: pts.append((float(nums[0]), float(nums[1]), float(nums[2])))
                elif len(nums) == 2: pts.append((float(nums[0]), float(nums[1]), float('nan')))
            if pts: strokes.append(pts)
        elif tag == 'annotation':
            typ = el.get('type', '')
            if typ == 'normalizedLabel': norm = el.text
            elif typ in ('label', 'truth'): raw = el.text
    label = (norm or raw or '').strip().strip('$').strip()
    return (None, None) if (bad or not strokes or not label) else (strokes, label)

def class_for_math_label(label):
    compact = label.replace(' ', '')
    if re.fullmatch(r'\d+([.,]\d+)?', compact):
        return 'NUMBER'
    if re.fullmatch(r'[A-Za-z\s]+', label):
        return None                     # purely alphabetic: ambiguous with TEXT, drop
    return 'MATH'

In [ ]:
# 3. Gather REAL strokes (HARD LANDING: NO SYNTHETIC FALLBACK)
samples = []        # dict(arr, label, split, group)
source_counts = {}
rng = random.Random(SEED)

# Path A: pre-packaged router_data.jsonl (your own labelled real data)
custom = [p for p in ['/kaggle/input/netz-router-data/router_data.jsonl', '/kaggle/input/router-data/router_data.jsonl',
                      '/kaggle/input/netz-strokes/router_data.jsonl', 'data/router_data.jsonl', 'router_data.jsonl'] if os.path.exists(p)]
for p in custom:
    n = 0
    with open(p, 'r', encoding='utf-8') as fh:
        for i, line in enumerate(fh):
            if not line.strip(): continue
            try:
                e = json.loads(line)
                arr = strokes_to_array(e['strokes'])
                lab = str(e.get('label', 'UNKNOWN')).upper()
                if len(arr) >= 2 and lab in CLASS_TO_IDX:
                    samples.append(dict(arr=arr, label=lab, split=hash_split(f'{p}:{i}'), group=f'custom:{i}')); n += 1
            except Exception:
                continue
    source_counts[f'custom:{p}'] = n
    print(f'[Real Data] {n} samples from {p}')

# Path B: harvest from attached IAM-OnDB (TEXT) and MathWriting/CROHME (NUMBER / MATH)
def collect_files(extra=()):
    roots = [r for r in [os.environ.get('NETZ_IAM_DIR'), os.environ.get('NETZ_MATH_DIR'), '/kaggle/input', 'data'] if r and os.path.isdir(r)] + list(extra)
    iam, mth = [], []
    for root in roots:
        for dp, _, files in os.walk(root):
            for f in files:
                if LINE_XML_RE.match(f):
                    iam.append(os.path.join(dp, f))
                elif f.lower().endswith('.inkml'):
                    mth.append(os.path.join(dp, f))
    return roots, iam, mth

roots, iam_files, math_files = collect_files()
if not iam_files and not math_files:
    roots, iam_files, math_files = collect_files(extract_archives(roots))        # maybe the data is zipped
if not iam_files:
    dl = download_iam()                                                          # YOUR private Kaggle dataset / URLs (optional)
    if dl:
        roots, iam_files, _m = collect_files(dl + extract_archives(dl))
if not math_files:
    mw = download_mathwriting()                                                  # official MathWriting, needs Internet ON
    if mw:
        _, _, mth_dl = collect_files([mw])
        math_files = list(mth_dl)
if not iam_files and not math_files:
    list_inputs()
iam_files, math_files = sorted(set(iam_files)), sorted(set(math_files))
math_files = [p for p in math_files if path_split_math(p) != 'EXCLUDED']
rng.shuffle(iam_files); rng.shuffle(math_files)
print(f'Found {len(iam_files)} IAM line files, {len(math_files)} math InkML files (synthetic/symbols excluded)')
if not iam_files:
    print('  -> no IAM-OnDB line strokes: the TEXT class cannot be built.\n' + IAM_HELP)
if not math_files:
    print('  -> no MathWriting/CROHME .inkml files: NUMBER/MATH classes cannot be built. Attach MathWriting.')

n = 0
for p in iam_files:
    if n >= CFG['per_class']: break
    try:
        arr = strokes_to_array(parse_iam_xml(p))
    except Exception:
        continue
    if len(arr) < 2: continue
    form = LINE_XML_RE.match(os.path.basename(p)).group(1).lower()
    samples.append(dict(arr=arr, label='TEXT', split=hash_split(form), group=form)); n += 1
source_counts['IAM-OnDB TEXT'] = n

official = any(path_split_math(p) in ('train', 'test') for p in math_files)
cnt = {'NUMBER': 0, 'MATH': 0}
scanned = dropped_alpha = 0
for p in math_files:
    if (cnt['NUMBER'] >= CFG['per_class'] and cnt['MATH'] >= CFG['per_class']) or scanned >= CFG['max_math_scan']: break
    scanned += 1
    sp = path_split_math(p) if official else hash_split(p)
    if sp is None: continue
    try:
        strokes, label = parse_inkml(p)
    except Exception:
        continue
    if strokes is None: continue
    cls = class_for_math_label(label)
    if cls is None:
        dropped_alpha += 1; continue
    if cnt[cls] >= CFG['per_class']: continue
    arr = strokes_to_array(strokes)
    if len(arr) < 2: continue
    samples.append(dict(arr=arr, label=cls, split=sp, group=p)); cnt[cls] += 1
source_counts['MathWriting NUMBER'], source_counts['MathWriting MATH'] = cnt['NUMBER'], cnt['MATH']
print(f'MathWriting: scanned {scanned} files, kept {cnt}, dropped {dropped_alpha} purely-alphabetic labels')


# Real OBJECT strokes (doodles of everyday objects) from Google Quick, Draw! - official public files, streamed (only the first lines of each file)
QD_BASE = os.environ.get('NETZ_QD_BASE', 'https://storage.googleapis.com/quickdraw_dataset/full/raw')
QD_CATEGORIES = ['cat', 'dog', 'house', 'tree', 'car', 'bicycle', 'flower', 'sun', 'star', 'fish', 'bird', 'apple', 'cup',
                 'chair', 'clock', 'moon', 'key', 'guitar', 'hat', 'umbrella', 'airplane', 'banana', 'camera', 'sailboat']

def load_quickdraw(n_total):
    import urllib.request
    per_cat = max(10, n_total // len(QD_CATEGORIES))
    out = []
    for cat in QD_CATEGORIES:
        got = 0
        try:
            with urllib.request.urlopen(f'{QD_BASE}/{cat}.ndjson', timeout=60) as resp:
                for raw in resp:
                    if got >= per_cat:
                        break
                    try:
                        rec = json.loads(raw)
                        if not rec.get('recognized', True):
                            continue
                        strokes = [[(x, y, t) for x, y, t in zip(st[0], st[1], st[2])] for st in rec['drawing'] if len(st) >= 3]
                        arr = strokes_to_array(strokes)
                    except Exception:
                        continue
                    if len(arr) < 4:
                        continue
                    kid = str(rec.get('key_id', f'{cat}{got}'))
                    out.append(dict(arr=arr, label='OBJECT', split=hash_split(kid), group='qd:' + kid)); got += 1
        except Exception as e:
            print(f'  Quick Draw "{cat}" skipped: {e}')
    return out

if CFG['quickdraw'] and not any(s['label'] == 'OBJECT' for s in samples):
    qd_samples = load_quickdraw(CFG['per_class'])
    samples.extend(qd_samples)
    source_counts['QuickDraw OBJECT'] = len(qd_samples)
    print(f'Quick, Draw!: {len(qd_samples)} real OBJECT drawings' + ('' if qd_samples else ' (none - is Internet ON?)'))

if len(samples) == 0:
    raise FileNotFoundError('\n' + '=' * 70 + '\n[FATAL - HARD LANDING]\nNo real stroke data found. Attach IAM-OnDB and/or MathWriting '
                            '(or your own router_data.jsonl). Synthetic fallback is disabled.\n' + '=' * 70)
print('Source counts:', source_counts)

In [ ]:
# 4. Optional composed MIXED class (OFF by default; composed from real strokes, not synthetic)
def compose_mixed(a, b, rng):
    """Place real math ink `b` to the right of / below real text ink `a`. Output arr = [x,y,t,stroke_id]."""
    a, b = a.copy(), b.copy()
    sa = max(np.ptp(a[:, 1]), 1e-6); sb = max(np.ptp(b[:, 1]), 1e-6)
    b[:, :2] = (b[:, :2] - b[:, :2].min(0)) * (sa / sb) * rng.uniform(0.8, 1.4)
    a[:, :2] -= a[:, :2].min(0)
    gap = 0.4 * sa
    if rng.random() < 0.5:
        b[:, 0] += a[:, 0].max() + gap
    else:
        b[:, 1] += a[:, 1].max() + gap
    b[:, 2] = np.where(np.isfinite(b[:, 2]), b[:, 2], 0) + (np.nanmax(a[:, 2]) if np.isfinite(a[:, 2]).any() else 0) + 800
    b[:, 3] += a[:, 3].max() + 1
    return np.vstack([a, b])

if CFG['compose_mixed']:
    texts = [s for s in samples if s['label'] == 'TEXT']
    maths = [s for s in samples if s['label'] == 'MATH']
    if texts and maths:
        crng = np.random.default_rng(SEED)
        k = 0
        for _ in range(min(CFG['per_class'], len(texts), len(maths))):
            t, m = texts[crng.integers(len(texts))], maths[crng.integers(len(maths))]
            if t['split'] != m['split']:
                continue                                              # never mix splits
            ta = t['arr'][t['arr'][:, 3] < crng.integers(3, 15)]     # a short real text crop
            if len(ta) < 2: continue
            samples.append(dict(arr=compose_mixed(ta, m['arr'], crng), label='MIXED', split=t['split'], group=t['group'] + '+' + m['group'])); k += 1
        print(f'[Opt-in] composed {k} MIXED samples from real text + real math crops')
else:
    print('MIXED composition disabled (default).')

In [ ]:
# 5. Class inventory, splits, honest class masking
present = sorted({s['label'] for s in samples}, key=lambda c: CLASS_TO_IDX[c])
print('Classes with real data:', present)
absent = [c for c in ROUTER_CLASSES if c not in present]
if absent:
    print(f'NOTE: no real data for {absent}. They are masked out of the exported model (never predicted). '
          'Provide router_data.jsonl with those labels to enable them.')
if len(present) < 2:
    raise RuntimeError('Need at least two classes with real data to train a router.')
missing_core = [c for c in ('TEXT', 'MATH') if c not in present]
if missing_core and os.environ.get('NETZ_ALLOW_PARTIAL', '0') != '1':
    raise RuntimeError('\n' + '=' * 70 + f'\n[HARD LANDING] No real data for core classes {missing_core}. A router that can never answer '
                       f'{missing_core} is useless in the app.\n' + IAM_HELP + '\n(Set NETZ_ALLOW_PARTIAL=1 to train a partial router anyway.)\n' + '=' * 70)

split_data = {k: [s for s in samples if s['split'] == k] for k in ('train', 'val', 'test')}
for k, v in split_data.items():
    c = {cl: sum(1 for s in v if s['label'] == cl) for cl in present}
    print(f'{k:5s}: {len(v):6d}  {c}')
    if len(v) == 0 or any(x == 0 for x in c.values()):
        raise RuntimeError(f'Split "{k}" is missing a class - not enough real data for an honest evaluation.')

In [ ]:
# 6. Dataset with stroke-crop + geometric augmentation
def crop_strokes(arr, rng, mode):
    sids = np.unique(arr[:, 3])
    ns = len(sids)
    if ns <= 3:
        return arr
    k = int(rng.integers(3, ns + 1))
    start = 0 if mode == 'prefix' else int(rng.integers(0, ns - k + 1))
    keep = np.isin(arr[:, 3], sids[start:start + k])
    out = arr[keep].copy()
    out[:, 3] = np.unique(out[:, 3], return_inverse=True)[1]
    return out

class RouterData(Dataset):
    def __init__(self, data, mode):
        """mode: 'train' (random crops+aug) | 'full' (clean) | 'crop' (deterministic partial ink)"""
        self.data, self.mode, self.rng = data, mode, None
    def __len__(self):
        return len(self.data)
    def __getitem__(self, i):
        s = self.data[i]
        arr = s['arr']
        if self.mode == 'train':
            if self.rng is None:
                self.rng = np.random.default_rng()
            r = self.rng.random()
            if r < 0.30: arr = crop_strokes(arr, self.rng, 'prefix')
            elif r < 0.55: arr = crop_strokes(arr, self.rng, 'window')
            arr = augment_array(arr, self.rng, thin=True)
        elif self.mode == 'crop':
            r = np.random.default_rng(10_000 + i)
            arr = crop_strokes(arr, r, 'prefix' if i % 2 == 0 else 'window')
        feats = extract_features(arr, CFG['max_len'], CFG['scale_mode'])
        return torch.from_numpy(feats), CLASS_TO_IDX[s['label']]

def collate(batch):
    feats, ys = zip(*batch)
    lengths = torch.tensor([len(f) for f in feats], dtype=torch.long)
    x = torch.zeros(len(feats), int(lengths.max()), FEATURE_DIM)
    for i, f in enumerate(feats):
        x[i, :len(f)] = f
    return x, lengths, torch.tensor(ys, dtype=torch.long)

def loader(split, mode, shuffle):
    return DataLoader(RouterData(split_data[split], mode), batch_size=CFG['batch_size'], shuffle=shuffle,
                      collate_fn=collate, num_workers=CFG['num_workers'], drop_last=(mode == 'train'))

train_loader = loader('train', 'train', True)
val_loader = loader('val', 'full', False)
test_loader_full, test_loader_crop = loader('test', 'full', False), loader('test', 'crop', False)

In [ ]:
# 7. Sanity check on real samples (mandatory)
print('--- [Sanity check] ---')
ds = RouterData(split_data['train'], 'full')
cts = {c: 0 for c in ROUTER_CLASSES}
for i in range(min(300, len(ds))):
    f, y = ds[i]
    assert f.ndim == 2 and f.shape[1] == FEATURE_DIM and torch.isfinite(f).all()
    cts[ROUTER_CLASSES[y]] += 1
print('label mix in first samples:', {k: v for k, v in cts.items() if v})
print('--- [Sanity check passed] ---')

In [ ]:
# 8. Model: geometry-only BiGRU + attention pooling (timing columns are masked inside the model)
GEOM_COLS = [0, 1, 2, 3, 5, 6]     # x, y, dx, dy, stroke_end, stroke_start  (dt=4 and speed=7 are NOT used)

class TemporalStrokeRouter(nn.Module):
    def __init__(self, d_model=96, num_layers=2, num_classes=len(ROUTER_CLASSES), dropout=0.2):
        super().__init__()
        self.register_buffer('cols', torch.tensor(GEOM_COLS, dtype=torch.long))
        self.register_buffer('class_bias', torch.zeros(num_classes))          # -30 for classes without real data
        self.in_proj = nn.Sequential(nn.Linear(len(GEOM_COLS), d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.gru = nn.GRU(d_model, d_model // 2, num_layers=num_layers, batch_first=True, bidirectional=True, dropout=dropout)
        self.attn = nn.Sequential(nn.Linear(d_model, d_model // 2), nn.Tanh(), nn.Linear(d_model // 2, 1))
        self.classifier = nn.Sequential(nn.Linear(d_model, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, num_classes))

    def forward(self, x, lengths=None):
        h = self.in_proj(x.index_select(-1, self.cols))
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(h, lengths.cpu(), batch_first=True, enforce_sorted=False)
            out, _ = self.gru(packed)
            out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=h.size(1))
        else:
            out, _ = self.gru(h)
        scores = self.attn(out).squeeze(-1)
        if lengths is not None:
            mask = torch.arange(out.size(1), device=out.device)[None, :] >= lengths.to(out.device)[:, None]
            scores = scores.masked_fill(mask, float('-inf'))
        w = F.softmax(scores, dim=1).unsqueeze(-1)
        return self.classifier((out * w).sum(1)) + self.class_bias

model = TemporalStrokeRouter().to(device)
bias = torch.full((len(ROUTER_CLASSES),), 0.0)
for c in absent:
    bias[CLASS_TO_IDX[c]] = -30.0
model.class_bias.copy_(bias.to(device))
print(f'Router: {sum(p.numel() for p in model.parameters()) / 1e3:.0f}K parameters | active classes: {present}')

In [ ]:
# 9. Training: class-weighted CE, AdamW + OneCycle, model selection on val MACRO-F1
def confusion(preds, targets, k=len(ROUTER_CLASSES)):
    cm = np.zeros((k, k), dtype=np.int64)
    for p, t in zip(preds, targets):
        cm[t, p] += 1
    return cm

def prf(cm):
    out = {}
    for c in present:
        i = CLASS_TO_IDX[c]
        tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i, :].sum() - tp
        p = tp / max(1, tp + fp); r = tp / max(1, tp + fn)
        out[c] = (p, r, 2 * p * r / max(1e-9, p + r))
    return out

def run_eval(ld):
    model.eval(); P, T = [], []
    with torch.no_grad():
        for x, lengths, y in ld:
            P.extend(model(x.to(device), lengths).argmax(-1).cpu().tolist()); T.extend(y.tolist())
    cm = confusion(P, T)
    m = prf(cm)
    return dict(acc=float((np.array(P) == np.array(T)).mean()), macro_f1=float(np.mean([v[2] for v in m.values()])), cm=cm, per_class=m)

counts = np.array([sum(1 for s in split_data['train'] if s['label'] == c) for c in ROUTER_CLASSES], dtype=np.float64)
weights = np.where(counts > 0, counts.sum() / np.maximum(counts, 1) / max(1, (counts > 0).sum()), 0.0)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=device), label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=CFG['lr'], total_steps=CFG['epochs'] * max(1, len(train_loader)), pct_start=0.15)

best_f1, best_state, bad = -1.0, None, 0
for epoch in range(1, CFG['epochs'] + 1):
    model.train(); tot, nb = 0.0, 0
    for x, lengths, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x, lengths), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step(); scheduler.step()
        tot += loss.item(); nb += 1
    v = run_eval(val_loader); flag = ''
    if v['macro_f1'] > best_f1:
        best_f1, best_state, bad, flag = v['macro_f1'], copy.deepcopy(model.state_dict()), 0, '  <- best'
    else:
        bad += 1
    print(f'Epoch {epoch:02d}/{CFG["epochs"]} | train loss {tot / max(1, nb):.4f} | val acc {v["acc"] * 100:.2f}% | val macro-F1 {v["macro_f1"] * 100:.2f}%{flag}')
    if bad >= CFG['patience']:
        print('Early stopping.'); break
model.load_state_dict(best_state)

In [ ]:
# 10. Untouched test evaluation (full ink AND partial/cropped ink) + shortcut detector
def report(name, r):
    print(f'--- {name}: accuracy {r["acc"] * 100:.2f}% | macro-F1 {r["macro_f1"] * 100:.2f}%')
    for c, (p, rc, f1) in r['per_class'].items():
        print(f'    {c:7s} precision {p * 100:5.1f}%  recall {rc * 100:5.1f}%  F1 {f1 * 100:5.1f}%')
    idx = [CLASS_TO_IDX[c] for c in present]
    print('    confusion (rows=true, cols=pred), order:', present)
    print('   ', r['cm'][np.ix_(idx, idx)].tolist())

res_full, res_crop = run_eval(test_loader_full), run_eval(test_loader_crop)
print('=== Untouched Test Set ===')
report('FULL ink', res_full)
report('PARTIAL ink (stroke crops)', res_crop)

# Shortcut detector: can trivial stats alone (stroke count, point count, aspect ratio) separate the classes?
def trivial_stats(arr):
    w, h = float(np.ptp(arr[:, 0])), float(np.ptp(arr[:, 1]))
    ns = len(np.unique(arr[:, 3]))
    return [ns, len(arr), w / max(h, 1e-6), len(arr) / max(1, ns)]
shortcut_acc = None
try:
    from sklearn.ensemble import GradientBoostingClassifier
    Xtr = np.array([trivial_stats(s['arr']) for s in split_data['train']]); ytr = [CLASS_TO_IDX[s['label']] for s in split_data['train']]
    Xte = np.array([trivial_stats(s['arr']) for s in split_data['test']]); yte = [CLASS_TO_IDX[s['label']] for s in split_data['test']]
    shortcut_acc = float((GradientBoostingClassifier(n_estimators=60, max_depth=3, random_state=SEED).fit(Xtr, ytr).predict(Xte) == np.array(yte)).mean())
    print(f'Shortcut baseline (stroke count / points / aspect ratio only): {shortcut_acc * 100:.2f}% accuracy')
    if shortcut_acc >= res_full['acc'] - 0.05:
        print('WARNING: trivial statistics are almost as good as the network. The classes are separable by dataset artifacts '
              '(size/length), so real-app accuracy may be much lower. Add more varied real data (esp. your own pen input).')
except ImportError:
    print('sklearn not available - shortcut baseline skipped.')

if res_crop['macro_f1'] < res_full['macro_f1'] - 0.15:
    print('NOTE: accuracy drops a lot on partial ink. Route only after enough strokes (see meta min_strokes) or collect more partial data.')

In [ ]:
# 11. Production ONNX export + verification
model.eval().cpu()
dummy = torch.randn(1, 128, FEATURE_DIM, dtype=torch.float32)
onnx_path = 'netz_router.onnx'
kw = dict(export_params=True, opset_version=17, do_constant_folding=True, input_names=['strokes'], output_names=['logits'],
          dynamic_axes={'strokes': {0: 'batch_size', 1: 'sequence_length'}, 'logits': {0: 'batch_size'}})
try:
    torch.onnx.export(model, dummy, onnx_path, dynamo=False, **kw)
except TypeError:
    torch.onnx.export(model, dummy, onnx_path, **kw)
onnx.checker.check_model(onnx.load(onnx_path))
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
max_diff = 0.0
for T in (8, 50, 128, CFG['max_len']):
    xin = torch.randn(1, T, FEATURE_DIM)
    with torch.no_grad():
        ref = model(xin).numpy()
    out = sess.run(None, {'strokes': xin.numpy()})[0]
    max_diff = max(max_diff, float(np.abs(ref - out).max()))
print(f'ONNX vs PyTorch max |logit diff|: {max_diff:.2e}')
assert max_diff < 1e-3, 'ONNX export does not match PyTorch'

with open('netz_router_meta.json', 'w') as fh:
    json.dump({'classes': ROUTER_CLASSES, 'active_classes': present, 'max_len': CFG['max_len'], 'scale_mode': CFG['scale_mode'],
               'uses_feature_columns': GEOM_COLS, 'feature_spec': FEATURE_SPEC, 'min_strokes_recommended': 3,
               'test_accuracy_full': round(res_full['acc'], 4), 'test_macro_f1_full': round(res_full['macro_f1'], 4),
               'test_macro_f1_partial': round(res_crop['macro_f1'], 4), 'shortcut_baseline_acc': shortcut_acc}, fh, indent=2)
write_feature_test('netz_router_feature_test.json', CFG['max_len'], CFG['scale_mode'])
print(f'Exported {onnx_path} ({os.path.getsize(onnx_path) / 1024:.1f} KB) + netz_router_meta.json')